# openoppsdb manager

This notebook is connected to `wyattowalsh/openoppsdb`. Schedule it with a daily Kaggle cron cadence such as `0 6 * * *`. Each run installs OpenOpps from GitHub, copies the newest `/kaggle/input/**/openoppsdb.sqlite` snapshot into `/kaggle/working/openoppsdb/openoppsdb.sqlite`, runs `openopps sync --metrics-json`, captures status and coverage evidence for the private quality gate, prepares SQLite/CSV/Parquet artifacts, prunes private evidence from the upload directory, and deploys a new dataset version only after the quality gate passes.


In [ ]:
#@title Initialize
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from datetime import UTC, datetime
import urllib.request

DATASET_ID = os.environ.get(
    "OPENOPPS_KAGGLE_DATASET",
    "wyattowalsh/openoppsdb",
)
PACKAGE_SPEC = os.environ.get(
    "OPENOPPS_PACKAGE_SPEC",
    "git+https://github.com/wyattowalsh/openopps.git@main",
)
OUTPUT_DIR = Path(
    os.environ.get(
        "OPENOPPS_KAGGLE_OUTPUT_DIR",
        "/kaggle/working/openoppsdb",
    )
)
DB_PATH = OUTPUT_DIR / "openoppsdb.sqlite"
GENERATOR_SCRIPT = OUTPUT_DIR / "generate_kaggle_metadata.py"
CSV_DIR = "exports/csv"
PARQUET_DIR = "exports/parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
INPUT_DB_GLOB = "**/openoppsdb.sqlite"
GENERATOR_SCRIPT_URL = os.environ.get(
    "OPENOPPS_GENERATOR_SCRIPT_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/scripts/generate_kaggle_metadata.py",
)
DATASET_IMAGE_URL = os.environ.get(
    "OPENOPPS_DATASET_IMAGE_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/docs/public/social/openoppsdb.png",
)
OPENOPPS_SYNC_ENV_DEFAULTS = {
    "OPENOPPS_BOARD_CONCURRENCY": "80",
    "OPENOPPS_HTTP_TIMEOUT": "20",
    "OPENOPPS_JOB_ROUTE_TIMEOUT_SECONDS": "180",
    "OPENOPPS_MAX_CONNECTIONS": "120",
    "OPENOPPS_PROVIDER_CONCURRENCY": "80",
    "OPENOPPS_RETRY_ATTEMPTS": "2",
    "OPENOPPS_SOURCE_CONCURRENCY": "40",
    "OPENOPPS_SOURCE_FRESHNESS_SECONDS": "86400",
    "OPENOPPS_SOURCE_TIMEOUT_SECONDS": "120"
}
KAGGLE_SYNC_TIMEOUT_SECONDS = float(
    os.environ.get(
        "OPENOPPS_KAGGLE_SYNC_TIMEOUT_SECONDS",
        "3300",
    )
)
KAGGLE_CREDENTIALS_ERROR = (
    "Kaggle API credentials are required to publish openoppsdb. "
    "Configure KAGGLE_USERNAME and KAGGLE_KEY as Kaggle notebook secrets "
    "before running the manager."
)
KAGGLE_SECRET_LOOKUP_ERRORS: dict[str, str] = {}

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_kaggle_notebook_secrets() -> None:
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        KAGGLE_SECRET_LOOKUP_ERRORS["kaggle_secrets"] = type(exc).__name__
        print(f"Kaggle notebook secrets client unavailable: {type(exc).__name__}")
        return

    client = UserSecretsClient()

    if os.environ.get("KAGGLE_USERNAME"):
        print("KAGGLE_USERNAME already present in environment.")
    else:
        try:
            username = client.get_secret("KAGGLE_USERNAME")
        except Exception as exc:
            KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_USERNAME"] = type(exc).__name__
            print(
                "KAGGLE_USERNAME notebook secret lookup failed: "
                f"{type(exc).__name__}"
            )
        else:
            if isinstance(username, str) and username.strip():
                os.environ["KAGGLE_USERNAME"] = username.strip()
                print("KAGGLE_USERNAME loaded from Kaggle notebook secrets.")
            else:
                print("KAGGLE_USERNAME not found in Kaggle notebook secrets.")

    if os.environ.get("KAGGLE_KEY"):
        print("KAGGLE_KEY already present in environment.")
    else:
        try:
            key = client.get_secret("KAGGLE_KEY")
        except Exception as exc:
            KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_KEY"] = type(exc).__name__
            print(
                "KAGGLE_KEY notebook secret lookup failed: "
                f"{type(exc).__name__}"
            )
        else:
            if isinstance(key, str) and key.strip():
                os.environ["KAGGLE_KEY"] = key.strip()
                print("KAGGLE_KEY loaded from Kaggle notebook secrets.")
            else:
                print("KAGGLE_KEY not found in Kaggle notebook secrets.")

def has_kaggle_credentials() -> bool:
    load_kaggle_notebook_secrets()
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    token_path = os.environ.get("KAGGLE_API_V1_TOKEN_PATH")
    return bool(
        os.environ.get("KAGGLE_API_TOKEN")
        or (token_path and Path(token_path).expanduser().exists())
        or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
        or kaggle_json.exists()
    )

def require_kaggle_credentials() -> None:
    if not has_kaggle_credentials():
        if KAGGLE_SECRET_LOOKUP_ERRORS:
            details = ", ".join(
                f"{key}={value}"
                for key, value in sorted(KAGGLE_SECRET_LOOKUP_ERRORS.items())
            )
            raise RuntimeError(f"{KAGGLE_CREDENTIALS_ERROR} Lookup diagnostics: {details}")
        raise RuntimeError(KAGGLE_CREDENTIALS_ERROR)

def run(command: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env)

def run_json(
    command: list[str],
    output_path: Path,
    *,
    env: dict[str, str] | None = None,
    timeout_seconds: float | None = None,
) -> dict:
    print("+", " ".join(command), ">", output_path)
    try:
        completed = subprocess.run(
            command,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_seconds,
        )
    except subprocess.TimeoutExpired as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr, file=sys.stderr)
        timeout_label = (
            f"{timeout_seconds:g}" if timeout_seconds is not None else "unknown"
        )
        raise TimeoutError(
            f"Command exceeded {timeout_label}s: {' '.join(command)}"
        ) from exc
    if completed.returncode:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
        completed.check_returncode()
    data = json.loads(completed.stdout)
    output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")
    print(f"Wrote {output_path}")
    return data

def install_openopps() -> None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", PACKAGE_SPEC, "kaggle"])

def copy_latest_input_db() -> None:
    db_candidates = sorted(KAGGLE_INPUT_DIR.glob(INPUT_DB_GLOB))
    if db_candidates:
        source_db = max(db_candidates, key=lambda path: path.stat().st_mtime)
        shutil.copy2(source_db, DB_PATH)
        print(f"Copied prior OpenOpps DB snapshot from {source_db} to {DB_PATH}")
    else:
        print("No prior OpenOpps DB snapshot found; creating a new ledger.")

def download_dataset_assets() -> None:
    urllib.request.urlretrieve(GENERATOR_SCRIPT_URL, GENERATOR_SCRIPT)
    urllib.request.urlretrieve(DATASET_IMAGE_URL, OUTPUT_DIR / "dataset-cover-image.png")

def update_kaggle_dataset_file_metadata() -> None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    from kagglesdk.datasets.types.dataset_api_service import (
        ApiUpdateDatasetMetadataRequest,
    )
    from kagglesdk.datasets.types.dataset_types import (
        DatasetSettings,
        DatasetSettingsFile,
        DatasetSettingsFileColumn,
    )

    metadata_path = OUTPUT_DIR / "dataset-metadata.json"
    metadata = json.loads(metadata_path.read_text())
    resources = metadata.get("resources") or []
    if not resources:
        raise RuntimeError(f"No Kaggle resources found in {metadata_path}")

    api = KaggleApi()
    api.authenticate()

    settings = DatasetSettings()
    settings.title = str(metadata.get("title") or "")
    settings.subtitle = str(metadata.get("subtitle") or "")
    settings.description = str(metadata.get("description") or "")
    settings.is_private = bool(metadata.get("isPrivate", False))
    settings.licenses = [
        api._new_license(str(license_data["name"]))
        for license_data in metadata.get("licenses", [])
        if license_data.get("name")
    ]
    settings.keywords = [str(keyword) for keyword in metadata.get("keywords", [])]
    settings.expected_update_frequency = str(
        metadata.get("expectedUpdateFrequency") or "not specified"
    )
    settings.user_specified_sources = str(metadata.get("userSpecifiedSources") or "")
    settings.data = [
        _dataset_settings_file(
            resource,
            DatasetSettingsFile,
            DatasetSettingsFileColumn,
            base_dir=OUTPUT_DIR,
        )
        for resource in resources
    ]

    owner_slug, dataset_slug = str(metadata.get("id") or DATASET_ID).split("/", 1)
    request = ApiUpdateDatasetMetadataRequest()
    request.owner_slug = owner_slug
    request.dataset_slug = dataset_slug
    request.settings = settings

    with api.build_kaggle_client() as kaggle:
        response = kaggle.datasets.dataset_api_client.update_dataset_metadata(request)
    errors = getattr(response, "errors", None) or []
    if errors:
        raise RuntimeError(f"Kaggle dataset metadata update failed: {errors}")
    print(f"Updated Kaggle file metadata for {len(settings.data or [])} public files.")

def _dataset_settings_file(
    resource,
    dataset_settings_file_cls,
    dataset_settings_file_column_cls,
    *,
    base_dir=None,
):
    file_metadata = dataset_settings_file_cls()
    file_metadata.name = str(resource["path"])
    file_metadata.description = str(resource.get("description") or "")
    if base_dir is not None:
        file_path = Path(base_dir) / str(resource["path"])
        if file_path.exists():
            file_metadata.total_bytes = file_path.stat().st_size
    columns = []
    for field in resource.get("schema", {}).get("fields", []):
        column = dataset_settings_file_column_cls()
        column.name = str(field["name"])
        column.description = str(field.get("description") or "")
        column.type = str(field.get("type") or "")
        columns.append(column)
    file_metadata.columns = columns
    return file_metadata

require_kaggle_credentials()
install_openopps()
copy_latest_input_db()
download_dataset_assets()


In [ ]:
openopps_env = os.environ.copy()
openopps_env["OPENOPPS_DB_URL"] = f"sqlite:///{DB_PATH}"
openopps_env["OPENOPPS_CACHE_ENABLED"] = "false"
for key, value in OPENOPPS_SYNC_ENV_DEFAULTS.items():
    openopps_env.setdefault(key, value)

run(["openopps", "admin", "db", "init"], env=openopps_env)
print(f"OpenOpps sync timeout: {KAGGLE_SYNC_TIMEOUT_SECONDS:g}s")
sync_metrics = run_json(
    ["openopps", "sync", "--metrics-json"],
    OUTPUT_DIR / "sync_metrics.json",
    env=openopps_env,
    timeout_seconds=KAGGLE_SYNC_TIMEOUT_SECONDS,
)
status = run_json(
    ["openopps", "status", "--json"],
    OUTPUT_DIR / "status.json",
    env=openopps_env,
)
coverage = run_json(
    ["openopps", "providers", "coverage", "--json"],
    OUTPUT_DIR / "coverage.json",
    env=openopps_env,
)


In [ ]:
quality_command = [
    sys.executable,
    str(GENERATOR_SCRIPT),
    "--output-dir",
    str(OUTPUT_DIR),
    "--data-db",
    str(DB_PATH),
    "--manager-dir",
    str(OUTPUT_DIR / "_manager-unused"),
    "--sync-metrics",
    str(OUTPUT_DIR / "sync_metrics.json"),
    "--status-json",
    str(OUTPUT_DIR / "status.json"),
    "--coverage-json",
    str(OUTPUT_DIR / "coverage.json"),
    "--quality-report",
    str(OUTPUT_DIR / "snapshot-quality.json"),
    "--prune-private-upload-files",
]
empty_snapshot_explanation = os.environ.get("OPENOPPS_EMPTY_SNAPSHOT_EXPLANATION")
if empty_snapshot_explanation:
    quality_command.extend([
        "--empty-snapshot-explanation",
        empty_snapshot_explanation,
    ])
run(quality_command)
shutil.rmtree(OUTPUT_DIR / "_manager-unused", ignore_errors=True)

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.name == "generate_kaggle_metadata.py":
        path.unlink()
        continue
    print(path.name, path.stat().st_size)


In [ ]:
message = f"Scheduled OpenOpps active-job snapshot {datetime.now(UTC).isoformat()}"
require_kaggle_credentials()

run([
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(OUTPUT_DIR),
    "-m",
    message,
    "-q",
    "-t",
    "-r",
    "zip",
])
update_kaggle_dataset_file_metadata()
run(["kaggle", "datasets", "status", DATASET_ID, "--format", "json"])
run(["kaggle", "datasets", "files", DATASET_ID, "--page-size", "200"])
